# GPT-1 (Radford et al., 2018): fine-tuning pipeline on Colab
Aditya Ware, Roll No. 26215011118, M.Tech-AI, MANIT

Runs the whole pipeline end to end on a Colab T4 (16 GB):
weight-match test → smoke tests → E1–E6 → tables/figures → `results.zip`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then upload `gpt1-assignment.zip`
(the repo folder zipped, without `cache/` and `checkpoints/`) with the file panel, or clone it from your own git remote.

In [ ]:
!nvidia-smi
import os
if not os.path.exists('gpt1-assignment/src'):
    if not os.path.exists('gpt1-assignment.zip'):
        raise SystemExit('Upload gpt1-assignment.zip first (left file panel), then re-run this cell.')
    !unzip -q -o gpt1-assignment.zip -d _unz
    # works for a zip of the folder itself and for a zip of its contents (PowerShell Compress-Archive)
    src = '_unz/gpt1-assignment' if os.path.exists('_unz/gpt1-assignment/src') else '_unz'
    !mv {src} gpt1-assignment
%cd gpt1-assignment

## 1. Install
Colab already has torch with CUDA. We add the pinned transformers version (still ships `OpenAIGPTModel`), plus ftfy and spaCy so the tokenizer reproduces the original GPT pre-processing.

In [ ]:
!pip -q install "transformers==4.57.6" "datasets>=3" ftfy spacy scikit-learn matplotlib tqdm
!python -m spacy download en_core_web_sm -q
import torch, transformers; print(torch.__version__, torch.cuda.get_device_name(0), transformers.__version__)

## 2. Verify the from-scratch model against the released weights
Asserts max |h_ours − h_HF| < 1e-4.

In [ ]:
!python src/load_pretrained.py --test

## 3. Smoke tests (100 examples, 1 epoch per script), with progress bars
Every script runs once on a tiny subset, so a bug surfaces in minutes rather than after an hour.

In [ ]:
!python src/run_all.py --smoke

## 4. The full experiment suite
`run_all.py` prints the plan with time estimates, then runs E1–E6 one after another. Each run shows its own
progress bar (optimizer steps, loss, accuracy, ETA). If Colab disconnects, re-run the cell: finished experiments
(`results/<name>.json`) are skipped.

A T4 has 16 GB, so `--micro-batch 32` (accumulation 1) is fastest. The effective batch stays 32 as in the paper.
Use 8 to reproduce the 4 GB-laptop configuration exactly.

In [ ]:
!python src/run_all.py --dry-run --micro-batch 32

In [ ]:
!python src/run_all.py --micro-batch 32

## 5. Look at a few outputs and classify your own text

In [ ]:
!python src/predict.py "the battery died after two days and support never replied" "absolutely love it, worth every penny"
from IPython.display import Image, display
import os
for f in ['comparison.png', 'curves_E1_sst2_seed42.png', 'layer_transfer_mrpc.png', 'attention_sst2.png']:
    p = 'results/figures/' + f
    if os.path.exists(p): display(Image(p))

## 6. Bring the results back
Download `results.zip` and unzip it over the local `gpt1-assignment/results/` folder, then run `python src/build_report.py` locally (see the README).

In [ ]:
!zip -qr results.zip results report/report.md report/report.docx
from google.colab import files; files.download('results.zip')